In [19]:
import os
import pandas as pd
import xarray as xr
import numpy as np

In [14]:
# Make sure logged in
import copernicusmarine
copernicusmarine.login()

INFO - 2026-03-20T22:25:14Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  eholmes


Copernicus Marine password:

  ········


File /home/jovyan/.copernicusmarine/.copernicusmarine-credentials already exists, overwrite it ? [y/N]:

  y


INFO - 2026-03-20T22:25:45Z - Credentials file stored in /home/jovyan/.copernicusmarine/.copernicusmarine-credentials.


True

In [20]:
import pandas as pd
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/point-collocation/main/"
    "examples/fixtures/points.csv"
)
df_points = pd.read_csv(url)
print(len(df_points))

# Let's add on our own pc_id column
df_points = df_points.reset_index(drop=True)
df_points["pc_id"] = df_points.index + 1
df_points["pc_label"] = "pace_" + df_points["pc_id"].astype(str)
#df_points = clean_dataset(df_points)
df_points.head()

595


,lat,lon,date,pc_id,pc_label
0,27.3835,-82.7375,2024-06-13,1,pace_1
1,27.1190,-82.7125,2024-06-14,2,pace_2
2,26.9435,-82.8170,2024-06-14,3,pace_3
3,26.6875,-82.8065,2024-06-14,4,pace_4
4,26.6675,-82.6455,2024-06-14,5,pace_5


In [2]:
import copernicusmarine as cm
dataset_id = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
cm.describe(dataset_id=dataset_id)

Fetching catalogue 1: 100%|██████████| 2/2 [00:01<00:00,  1.22it/s]


CopernicusMarineCatalogue(products=[CopernicusMarineProduct(title='Global Ocean Physics Reanalysis', product_id='GLOBAL_MULTIYEAR_PHY_001_030', thumbnail_url='https://mdl-metadata.s3.waw3-1.cloudferro.com/metadata/thumbnails/GLOBAL_MULTIYEAR_PHY_001_030.jpg', description='The GLORYS12V1 product is the CMEMS global ocean eddy-resolving (1/12° horizontal resolution, 50 vertical levels) reanalysis covering the altimetry (1993 onward).\n\nIt is based largely on the current real-time global forecasting CMEMS system. The model component is the NEMO platform driven at surface by ECMWF ERA-Interim then ERA5 reanalyses for recent years. Observations are assimilated by means of a reduced-order Kalman filter. Along track altimeter data (Sea Level Anomaly), Satellite Sea Surface Temperature, Sea Ice Concentration and In situ Temperature and Salinity vertical Profiles are jointly assimilated. Moreover, a 3D-VAR scheme provides a correction for the slowly-evolving large-scale biases in temperature a

In [27]:
import copernicusmarine
dataset_id = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
ds = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)
variables = ['uo','vo']
selected_variables_dataset = ds[variables]
selected_variables_dataset

INFO - 2026-03-20T22:47:28Z - Selected dataset version: "202311"
INFO - 2026-03-20T22:47:28Z - Selected dataset part: "default"


<xarray.Dataset> Size: 85TB
Dimensions:    (time: 12108, depth: 50, latitude: 2041, longitude: 4320)
Coordinates:
  * time       (time) datetime64[ns] 97kB 1993-01-01 1993-01-02 ... 2026-02-24
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92 90.0
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    uo         (time, depth, latitude, longitude) float64 43TB ...
    vo         (time, depth, latitude, longitude) float64 43TB ...
Attributes:
    comment:      CMEMS product
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    source:       MERCATOR GLORYS12V1
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    institution:  MERCATOR OCEAN
    references:   http://www.mercator-ocean.fr
    Conventions:  CF-1.4

In [33]:
# Copy input dataframe
df = df_points[0:10].copy()

# Build vectorized indexers
time_points = xr.DataArray(pd.to_datetime(df["date"]).to_numpy(), dims="points")
latitude_points = xr.DataArray(df["lat"].to_numpy(dtype=float), dims="points")
longitude_points = xr.DataArray(df["lon"].to_numpy(dtype=float), dims="points")

selection_indexers = {
    "time": time_points,
    "latitude": latitude_points,
    "longitude": longitude_points,
}
selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")
selected_points_dataset = selected_points_dataset.isel(depth=0)

downloaded_variables_dataframe = selected_points_dataset.to_dataframe().reset_index(drop=True)

# Add variables to input dataframe
out = pd.concat(
    [df.reset_index(drop=True), downloaded_variables_dataframe.reset_index(drop=True)],
    axis=1,
)

out

,lat,lon,date,pc_id,pc_label,uo,vo,depth,latitude,longitude,time
0,27.3835,-82.7375,2024-06-13,1,pace_1,-0.043947,0.085452,0.494025,27.416666,-82.750000,2024-06-13
1,27.1190,-82.7125,2024-06-14,2,pace_2,-0.070193,0.180670,0.494025,27.083334,-82.750000,2024-06-14
2,26.9435,-82.8170,2024-06-14,3,pace_3,-0.025025,0.117191,0.494025,26.916666,-82.833336,2024-06-14
3,26.6875,-82.8065,2024-06-14,4,pace_4,0.022584,0.088504,0.494025,26.666666,-82.833336,2024-06-14
4,26.6675,-82.6455,2024-06-14,5,pace_5,0.010376,0.125126,0.494025,26.666666,-82.666664,2024-06-14
5,26.8985,-82.6710,2024-06-14,6,pace_6,-0.074465,0.202033,0.494025,26.916666,-82.666664,2024-06-14
6,26.8965,-82.5660,2024-06-14,7,pace_7,-0.104373,0.212409,0.494025,26.916666,-82.583336,2024-06-14
7,26.8925,-82.4220,2024-06-14,8,pace_8,-0.116581,0.181280,0.494025,26.916666,-82.416664,2024-06-14
8,26.7135,-82.4535,2024-06-14,9,pace_9,-0.099490,0.188604,0.494025,26.750000,-82.416664,2024-06-14
9,26.5020,-82.3010,2024-06-14,10,pace_10,-0.007935,0.186773,0.494025,26.500000,-82.333336,2024-06-14


In [34]:
selected_points_dataset

<xarray.Dataset> Size: 324B
Dimensions:    (points: 10)
Coordinates:
    depth      float32 4B 0.494
    latitude   (points) float32 40B 27.42 27.08 26.92 26.67 ... 26.92 26.75 26.5
    longitude  (points) float32 40B -82.75 -82.75 -82.83 ... -82.42 -82.33
    time       (points) datetime64[ns] 80B 2024-06-13 2024-06-14 ... 2024-06-14
Dimensions without coordinates: points
Data variables:
    uo         (points) float64 80B ...
    vo         (points) float64 80B ...
Attributes:
    comment:      CMEMS product
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    source:       MERCATOR GLORYS12V1
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    institution:  MERCATOR OCEAN
    references:   http://www.mercator-ocean.fr
    Conventions:  CF-1.4

import pandas as pd
import numpy as np

# number of points
n = 100

# create dataframe
input_dataframe = pd.DataFrame({
    "LATITUDE": np.random.uniform(40, 42, n),
    "LONGITUDE": np.random.uniform(4, 8, n),
    "START": pd.to_datetime("2020-01-01"),
    "END": pd.to_datetime("2020-12-31"),
    "DEPTH": np.random.uniform(0, 10, n),
    "TIME": pd.to_datetime("2020-01-01")
})

print(input_dataframe.head())

In [30]:
len(df)

10

In [2]:
def clean_dataset(dataset):
    """
    Rename dimensions with standard names and sort values if appropriate
    """
    # Rename dimensions if incorrect
    for coordinate in dataset.coords:
       if coordinate=='lon':
           dataset = dataset.rename({'lon': 'longitude'})
       if coordinate=='lat':
           dataset = dataset.rename({'lat': 'latitude'})
       if coordinate=='DEPTH':
           dataset = dataset.rename({'lat': 'depth'})
          
    # Sort correctly lon/lat if they were inverted
    for dimension in ["latitude","longitude"]:
        coordinates = dataset[dimension].values
        if (coordinates[0] >= coordinates[:-1]).all():
            dataset = dataset.sortby(dimension, ascending=True)
        
    return dataset

In [6]:
# List of Copernicus Marine datasets (datasetIDs)
list_datasets = [
    'cmems_mod_med_phy-cur_my_4.2km_P1D-m',
]

# Output result names 
output_names = [
    'current_006_004',
]

# Variables to download
variables = ['uo','vo']

# SINGLE DATES ONLY: Set to 'True' if you want to keep depth/lat/lon/time from dataset (for checking purpose)
include_dataset_columns = True

# TIMESERIES ONLY: Set to 'True' to save output files in NetCDF format instead of CSV
save_as_netcdf = False


In [11]:
import copernicusmarine
dataset_id = "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.25deg_P1D"
ds = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)
ds

INFO - 2026-03-20T22:03:49Z - Selected dataset version: "202311"
INFO - 2026-03-20T22:03:49Z - Selected dataset part: "default"


<xarray.Dataset> Size: 65GB
Dimensions:    (time: 784, latitude: 720, longitude: 1440, nv: 2)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2022-10-04 2022-10-05 ... 2024-11-25
  * latitude   (latitude) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * longitude  (longitude) float32 6kB -179.9 -179.6 -179.4 ... 179.6 179.9
  * nv         (nv) int32 8B 0 1
    lat_bnds   (latitude, nv) float32 6kB ...
    lon_bnds   (longitude, nv) float32 12kB ...
Data variables:
    adt        (time, latitude, longitude) float64 7GB ...
    err_sla    (time, latitude, longitude) float64 7GB ...
    err_ugosa  (time, latitude, longitude) float64 7GB ...
    err_vgosa  (time, latitude, longitude) float64 7GB ...
    flag_ice   (time, latitude, longitude) float64 7GB ...
    sla        (time, latitude, longitude) float64 7GB ...
    ugos       (time, latitude, longitude) float64 7GB ...
    ugosa      (time, latitude, longitude) float64 7GB ...
    vgos       (time, latitude, longitude) float64 7GB ...
    vgosa      (time, latitude, longitude) float64 7GB ...
Attributes:
    comment:      Sea Surface Height measured by Altimetry and derived variables
    history:      2023-11-24 00:53:07Z: Creation
    source:       Altimetry measurements
    title:        NRT merged all satellites Global Ocean Gridded SSALTO/DUACS...
    contact:      servicedesk.cmems@mercator-ocean.eu
    institution:  CLS, CNES
    references:   http://marine.copernicus.eu
    Conventions:  CF-1.6

In [12]:
# Gloyrs
dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m"
ds = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)
ds

INFO - 2026-03-20T22:07:24Z - Selected dataset version: "202311"
INFO - 2026-03-20T22:07:24Z - Selected dataset part: "default"


<xarray.Dataset> Size: 177TB
Dimensions:    (depth: 50, latitude: 2041, longitude: 4320, time: 12108)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92 90.0
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
  * time       (time) datetime64[ns] 97kB 1993-01-01 1993-01-02 ... 2026-02-24
Data variables:
    bottomT    (time, latitude, longitude) float64 854GB ...
    mlotst     (time, latitude, longitude) float64 854GB ...
    siconc     (time, latitude, longitude) float64 854GB ...
    sithick    (time, latitude, longitude) float64 854GB ...
    so         (time, depth, latitude, longitude) float64 43TB ...
    thetao     (time, depth, latitude, longitude) float64 43TB ...
    uo         (time, depth, latitude, longitude) float64 43TB ...
    usi        (time, latitude, longitude) float64 854GB ...
    vo         (time, depth, latitude, longitude) float64 43TB ...
    vsi        (time, latitude, longitude) float64 854GB ...
    zos        (time, latitude, longitude) float64 854GB ...
Attributes:
    comment:      CMEMS product
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    source:       MERCATOR GLORYS12V1
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    institution:  MERCATOR OCEAN
    references:   http://www.mercator-ocean.fr
    Conventions:  CF-1.4

In [15]:
dataset_id="cmems_obs-sl_glo_phy-ssh_nrt_c2n-l3-duacs_PT1S"
ds = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)
ds

INFO - 2026-03-20T22:26:06Z - Selected dataset version: "202311"
INFO - 2026-03-20T22:26:06Z - Selected dataset part: "default"


TypeError: 'Command' object is not subscriptable

In [ ]:
copernicusmarine.subset(
  dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
  variables=["uo", "vo"],
  minimum_longitude=50,
  maximum_longitude=90,
  minimum_latitude=0,
  maximum_latitude=25,
  start_datetime="2020-01-01",
  end_datetime="2020-01-31",
  minimum_depth=0,
  maximum_depth=30,
)

In [7]:
for dataset_id, output_name in zip(list_datasets, output_names):

    # Read dataset via Toolbox
    dataset = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0) # chunk size limit set to 0 for single time steps

    # Clean dataset
    dataset = clean_dataset(dataset)

    # Select only requested variables first (important for correct depth detection)
    selected_variables_dataset = dataset[variables]

    # Copy input dataframe
    input_points_dataframe = input_dataframe.copy()

    # Build vectorized indexers
    time_points = xr.DataArray(pd.to_datetime(input_points_dataframe["TIME"]).to_numpy(), dims="points")
    latitude_points = xr.DataArray(input_points_dataframe["LATITUDE"].to_numpy(dtype=float), dims="points")
    longitude_points = xr.DataArray(input_points_dataframe["LONGITUDE"].to_numpy(dtype=float), dims="points")

    selection_indexers = {
        "time": time_points,
        "latitude": latitude_points,
        "longitude": longitude_points,
    }

    # Depth detection must be done on the selected variables dataset (not the full dataset)
    dataset_has_depth = "depth" in selected_variables_dataset.dims
    dataframe_has_depth = "DEPTH" in input_points_dataframe.columns and input_points_dataframe["DEPTH"].notna().any()

    # Download data for 3D variables
    if dataset_has_depth:
        if dataframe_has_depth:
            depth_points = xr.DataArray(input_points_dataframe["DEPTH"].to_numpy(dtype=float), dims="points")
            selection_indexers["depth"] = depth_points
            selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")
        else:
            selected_points_dataset = selected_variables_dataset.isel(depth=0).sel(**selection_indexers, method="nearest")
    else:
        # Download data for 2D variables
        selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")

    # Convert dataset to dataframe
    downloaded_variables_dataframe = selected_points_dataset.to_dataframe().reset_index(drop=True)

    # Identify index-like columns coming from xarray
    index_like_columns = [column for column in ("time", "latitude", "longitude", "depth") if column in downloaded_variables_dataframe.columns]

    # Keep depth/lat/lon/time from the dataset only if requested
    if not include_dataset_columns:
        downloaded_variables_dataframe = downloaded_variables_dataframe.drop(columns=index_like_columns, errors="ignore")

    # Add variables to input dataframe
    output_dataframe = pd.concat(
        [input_points_dataframe.reset_index(drop=True), downloaded_variables_dataframe.reset_index(drop=True)],
        axis=1,
    )

    output_dataframe

INFO - 2026-03-18T17:56:18Z - Selected dataset version: "202511"
INFO - 2026-03-18T17:56:18Z - Selected dataset part: "default"


In [8]:
output_dataframe

,LATITUDE,LONGITUDE,START,END,DEPTH,TIME,uo,vo,depth,latitude,longitude,time
0,41.836353,5.263667,2020-01-01,2020-12-31,6.575323,2020-01-01,-0.050474,-0.022055,5.464963,41.854168,5.250000,2020-01-01
1,41.320826,4.778977,2020-01-01,2020-12-31,3.133180,2020-01-01,0.296798,-0.036456,3.165747,41.312500,4.791667,2020-01-01
2,40.223916,7.108160,2020-01-01,2020-12-31,6.065819,2020-01-01,-0.035955,0.074415,5.464963,40.229168,7.125000,2020-01-01
3,40.837865,5.674847,2020-01-01,2020-12-31,2.860700,2020-01-01,0.126110,0.086549,3.165747,40.854168,5.666667,2020-01-01
4,40.396755,7.382790,2020-01-01,2020-12-31,4.942303,2020-01-01,0.131832,-0.137944,5.464963,40.395832,7.375000,2020-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...
95,41.262296,7.552853,2020-01-01,2020-12-31,0.313150,2020-01-01,-0.059806,0.139902,1.018237,41.270832,7.541667,2020-01-01
96,40.764744,4.223551,2020-01-01,2020-12-31,3.808210,2020-01-01,0.050801,0.055411,3.165747,40.770832,4.208333,2020-01-01
97,41.729007,7.744567,2020-01-01,2020-12-31,3.175710,2020-01-01,-0.024087,-0.275042,3.165747,41.729168,7.750000,2020-01-01
98,40.157205,4.462016,2020-01-01,2020-12-31,9.004257,2020-01-01,0.238996,-0.168046,7.920377,40.145832,4.458333,2020-01-01


In [11]:
dataset

<xarray.Dataset> Size: 6TB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 14304)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 114kB 1987-01-01 1987-01-02 ... 2026-02-28
Data variables:
    uo         (time, depth, latitude, longitude) float32 3TB ...
    vo         (time, depth, latitude, longitude) float32 3TB ...
Attributes:
    title:        Horizontal Velocity (3D) - Daily Mean
    source:       MFS E3R1I
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    contact:      servicedesk.cmems@mercator-ocean.eu
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    Conventions:  CF-1.0
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...

In [13]:
import numpy as np

lon_idx = np.linspace(0, dataset.sizes["longitude"] - 1, 10, dtype=int)
lat_idx = np.linspace(0, dataset.sizes["latitude"] - 1, 10, dtype=int)
depth_idx = np.linspace(0, dataset.sizes["depth"] - 1, 10, dtype=int)

sub = (
    dataset.sel(time=slice("2024-01-01", "2024-12-31"))
        .isel(depth=depth_idx)
        .isel(longitude=lon_idx, latitude=lat_idx)
)
time_idx = np.linspace(0, sub.sizes["time"] - 1, 20, dtype=int)
sub = sub.isel(time=time_idx)

In [14]:
sub.nbytes/1e6

0.16028

In [15]:
sub.to_netcdf("multi_time_depth.nc")

In [5]:
import copernicusmarine
dataset_id = 'cmems_mod_med_phy-cur_my_4.2km_P1D-m'
dataset = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)

INFO - 2026-03-18T20:53:53Z - Selected dataset version: "202511"
INFO - 2026-03-18T20:53:53Z - Selected dataset part: "default"


In [6]:
dataset

<xarray.Dataset> Size: 6TB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 14304)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 114kB 1987-01-01 1987-01-02 ... 2026-02-28
Data variables:
    uo         (time, depth, latitude, longitude) float32 3TB ...
    vo         (time, depth, latitude, longitude) float32 3TB ...
Attributes:
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...
    title:        Horizontal Velocity (3D) - Daily Mean
    contact:      servicedesk.cmems@mercator-ocean.eu
    Conventions:  CF-1.0
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       MFS E3R1I